In [0]:

null	false-- Total pledged per position (all pledgees combined)

pledge_agg AS (

    SELECT

        UPPER(TRIM(Cusip))           AS clean_cusip,

        CAST(As_Of_Date AS DATE)     AS clean_as_of_date,

        CAST(TransNo AS INTEGER)     AS clean_trans_no,

        SUM(PledgedMarketValue)      AS total_pledged_mv,

        COUNT(DISTINCT PledgeName)   AS num_pledgees

    FROM silver_${lakehouse.env}.tpg_reports.pg91_pledges_to_pledgee_portfolio_consolidate

    GROUP BY UPPER(TRIM(Cusip)), CAST(As_Of_Date AS DATE), CAST(TransNo AS INTEGER)

),
-- Total pledged per position (all pledgees combined)

pledge_agg AS (

    SELECT

        UPPER(TRIM(Cusip))           AS clean_cusip,

        CAST(As_Of_Date AS DATE)     AS clean_as_of_date,

        CAST(TransNo AS INTEGER)     AS clean_trans_no,

        SUM(PledgedMarketValue)      AS total_pledged_mv,

        COUNT(DISTINCT PledgeName)   AS num_pledgees

    FROM silver_${lakehouse.env}.tpg_reports.pg91_pledges_to_pledgee_portfolio_consolidate

    GROUP BY UPPER(TRIM(Cusip)), CAST(As_Of_Date AS DATE), CAST(TransNo AS INTEGER)

),
 
-- Pledged amount per pledgee (one row per PledgeName)

pledge_by_pledgee AS (

    SELECT

        UPPER(TRIM(Cusip))           AS clean_cusip,

        CAST(As_Of_Date AS DATE)     AS clean_as_of_date,

        CAST(TransNo AS INTEGER)     AS clean_trans_no,

        PledgeName,

        SUM(PledgedMarketValue)      AS pledgee_mv

    FROM silver_${lakehouse.env}.tpg_reports.pg91_pledges_to_pledgee_portfolio_consolidate

    GROUP BY UPPER(TRIM(Cusip)), CAST(As_Of_Date AS DATE), CAST(TransNo AS INTEGER), PledgeName

),
-- Pledged amount per pledgee (one row per PledgeName)

pledge_by_pledgee AS (

    SELECT

        UPPER(TRIM(Cusip))           AS clean_cusip,

        CAST(As_Of_Date AS DATE)     AS clean_as_of_date,

        CAST(TransNo AS INTEGER)     AS clean_trans_no,

        PledgeName,

        SUM(PledgedMarketValue)      AS pledgee_mv

    FROM silver_${lakehouse.env}.tpg_reports.pg91_pledges_to_pledgee_portfolio_consolidate

    GROUP BY UPPER(TRIM(Cusip)), CAST(As_Of_Date AS DATE), CAST(TransNo AS INTEGER), PledgeName

),
 
 
unpledged_safekeeping AS (

    SELECT

        CAST(TransNo AS INTEGER)     AS clean_trans_no,

        CAST(As_Of_Date AS DATE)     AS clean_as_of_date,

        ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(SafekeepingCode)), ', ') AS safekeeping_code  -- e.g. 'BNY, FRB'

    FROM silver_${lakehouse.env}.tpg_reports.pg93_safekeeping_by_location

    WHERE PledgeName = '<Unpledged>'

    GROUP BY CAST(TransNo AS INTEGER), CAST(As_Of_Date AS DATE)

),
 
-- Which TransNo to match on (2 hard-coded overrides, otherwise PositionID)

COALESCE(ptm.clean_trans_no, pg.clean_position_id) AS effective_trans_no,
 
-- Pledge status

CASE

    WHEN pg.FairValue = 0 OR pg.FairValue IS NULL          THEN 'ZERO_MV'

    WHEN pa.clean_trans_no IS NULL                         THEN 'NONE'      -- no pledge rows

    WHEN ABS(pa.total_pledged_mv - pg.FairValue) <= 50.0   THEN 'FULL'      -- within $50

    ELSE 'PARTIAL'

END AS pledge_status,
 
-- % of position that is pledged (max 100%)

CASE

    WHEN pg.FairValue IS NULL OR pg.FairValue = 0 THEN 0

    ELSE LEAST(COALESCE(pa.total_pledged_mv, 0) / pg.FairValue, 1.0)

END AS total_pledge_ratio
 
...

LEFT JOIN pledge_agg pa

    ON  pg.clean_cusip      = pa.clean_cusip

    AND pg.clean_as_of_date = pa.clean_as_of_date

    AND COALESCE(ptm.clean_trans_no, pg.clean_position_id) = pa.clean_trans_no
 
 
-- A) NOT PLEDGED (NONE / ZERO_MV): 1 row, 100%

tpg_none_zero AS (

    SELECT pc.*,

        1.0                  AS split_ratio,

        FALSE                AS split_pledged_flag,

        TRUE                 AS split_unencumbered,

        sk.safekeeping_code  AS split_pledged_to          -- safekeeping location

    FROM tpg_classified pc

    LEFT JOIN unpledged_safekeeping sk

        ON  pc.effective_trans_no = sk.clean_trans_no

        AND pc.clean_as_of_date   = sk.clean_as_of_date

    WHERE pc.pledge_status IN ('NONE', 'ZERO_MV')

),
 
-- B) FULLY PLEDGED: 1 row per pledgee

tpg_full AS (

    SELECT pc.*,

        pbp.pledgee_mv / NULLIF(pc.total_pledged_mv, 0) AS split_ratio,

        TRUE                                            AS split_pledged_flag,

        FALSE                                           AS split_unencumbered,

        CASE WHEN UPPER(pbp.PledgeName) = 'SYNOVUS REPOS' THEN 'To:'
ELSE If pbp.PledgeName Like (“Federal Home Loan”, “FHLB”, “FRB”, “BOE”, “BOC”, “BOJ”, “ECB”, “RBA”) then pbp.PledgeName
ELSE
Concatenate (“Other: ”, pbp.PledgeName)


             ELSE pbp.PledgeName END                    AS split_pledged_to   -- pledgee name

    FROM tpg_classified pc

    JOIN pledge_by_pledgee pbp

        ON  pc.clean_cusip        = pbp.clean_cusip

        AND pc.clean_as_of_date   = pbp.clean_as_of_date

        AND pc.effective_trans_no = pbp.clean_trans_no

    WHERE pc.pledge_status = 'FULL'

),
 
-- C1) PARTIALLY PLEDGED, pledged part: 1 row per pledgee

tpg_partial_pledged AS (

    SELECT pc.*,

        pbp.pledgee_mv / NULLIF(pc.FairValue, 0)        AS split_ratio,  -- note: ÷ FairValue

        TRUE                                            AS split_pledged_flag,

        FALSE                                           AS split_unencumbered,

        CASE WHEN UPPER(pbp.PledgeName) = 'SYNOVUS REPOS' THEN 'To:'
If pbp.PledgeName Like (“Federal Home Loan”, “FHLB”, “FRB”, “BOE”, “BOC”, “BOJ”, “ECB”, “RBA”) then pbp.PledgeName
ELSE
Concatenate (“Other: ”, pbp.PledgeName)


             ELSE pbp.PledgeName END                    AS split_pledged_to

    FROM tpg_classified pc

    JOIN pledge_by_pledgee pbp

        ON  pc.clean_cusip        = pbp.clean_cusip

        AND pc.clean_as_of_date   = pbp.clean_as_of_date

        AND pc.effective_trans_no = pbp.clean_trans_no

    WHERE pc.pledge_status = 'PARTIAL'

),
 
-- C2) PARTIALLY PLEDGED, leftover unpledged part: 1 row

tpg_partial_unpledged AS (

    SELECT pc.*,

        1 - pc.total_pledge_ratio  AS split_ratio,                -- remaining %

        FALSE                      AS split_pledged_flag,

        TRUE                       AS split_unencumbered,

        sk.safekeeping_code        AS split_pledged_to

    FROM tpg_classified pc

    LEFT JOIN unpledged_safekeeping sk

        ON  pc.effective_trans_no = sk.clean_trans_no

        AND pc.clean_as_of_date   = sk.clean_as_of_date

    WHERE pc.pledge_status = 'PARTIAL'

),
 
-- Combine all

tpg_split_all AS (

    SELECT * FROM tpg_none_zero

    UNION ALL SELECT * FROM tpg_full

    UNION ALL SELECT * FROM tpg_partial_pledged

    UNION ALL SELECT * FROM tpg_partial_unpledged

),
 
 
tpg_split_bucketed AS (

    SELECT * EXCEPT (split_pledged_to),

        CASE

            WHEN UPPER(TRIM(COALESCE(split_pledged_to, ''))) = 'FED' THEN 'FRB'

            ELSE split_pledged_to

        END AS split_pledged_to,

        ...

    FROM tpg_split_all

),
 
 
-- TPG select

CAST(s.split_pledged_to AS VARCHAR(250))  AS pledged_to,

s.split_pledged_flag                      AS pledged_flag,
 
-- NFS select

CAST(NULL AS VARCHAR(250))                AS pledged_to,

FALSE                                     AS pledged_flag,
 
 
https://snventerprise.sharepoint.com/sites/Teams-Axiom/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FTeams%2DAxiom%2FShared%20Documents%2FStrategic%202052A%20and%20CCAR%2FQA%20Alternate%20Data%20%2D%20Manual%20Upload%20by%20Business%2FTreasury%2FPledged%20Loans&viewid=544b8386%2Dad30%2D4ce3%2Dbf6a%2Db6bae5001f24&FolderCTID=0x012000FCFA7A4F9704D14598FF7B3C81939406
 
Mohan,Sreenivas, look at the profisee_sec_master file
 
Mohan,Sreenivas, our updates for the pledged to and pledged flag
pledged_to	pledged_flag

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

Other: HOUSING AUTHORITY FORT WALTON BEACH FL	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

null	false

Other: REPURCHASE AGREEMENT	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: RICHLAND COUNTY SC TREASURER	true

Other: STATE TREASURER OF ALABAMA	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

Other: HOUSING AUTHORITY OF BREMEN GA	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: CITY OF LORIS SC	true

Other: SANTEE-LYNCHES REGIONAL COUNCIL OF GOVERNMENTS	true

Other: RICHLAND COUNTY SC TREASURER	true

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: TOWN OF AWENDAW SOUTH CAROLINA	true

Other: STATE TREASURER OF FLORIDA	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: COLLIER'S INTERNATIONAL FBO CITY OF COLLEGE PARK SC	true

Other: HOUSING AUTHORITY OF BOWDON GEORGIA	true

Other: SANTEE COOPER AUTHORITY	true

Other: SOLID WASTE  DISPOSAL AUTHORITY - HUNTSVILLE AL	true

Other: HOUSING AUTHORITY CRESTVIEW FL	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: GEORGIA HAP ADMINISTRATION AKA NATIONAL HSG COMPLIANCE	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

null	false

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

null	false

null	false

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: STATE TREASURER OF ALABAMA	true

Other: KERSHAW HEALTH DBA THE HEALTH SERVICES DISTRICT OF KERSHAW CO	true

Other: GEORGIA HAP ADMINISTRATION AKA NATIONAL HSG COMPLIANCE	true

Other: SOUTH CAROLINA STATE TREASURER - DEDICATED	true

Other: TOWN OF AWENDAW SOUTH CAROLINA	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: STATE TREASURER OF ALABAMA	true

Other: REPURCHASE AGREEMENT	true

Other: REPURCHASE AGREEMENT	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: REPURCHASE AGREEMENT	true

Other: REPURCHASE AGREEMENT	true

null	false

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

Other: STATE TREASURER OF FLORIDA	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: CITY OF NORTH MYRTLE BEACH SC	true

Other: FLORENCE COUNTY SC TREASURER	true

Other: SANTEE COOPER AUTHORITY	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

Other: HOUSING AUTHORITY OF BREMEN GA	true

Other: STATE TREASURER OF FLORIDA	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: SOUTH CAROLINA STATE TREASURER - DEDICATED	true

Other: SPARTANBURG COUNTY SC PUBLIC LIBRARIES	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

Other: COLLIER'S INTERNATIONAL FBO CITY OF COLLEGE PARK SC	true

Other: SOUTH CAROLINA STATE TREASURER - DEDICATED	true

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF ALABAMA	true

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: HOUSING AUTHORITY FORT WALTON BEACH FL	true

null	false

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: SOUTH CAROLINA STATE PORTS AUTHORITY - MCA	true

Other: SPARTANBURG COUNTY SC PUBLIC LIBRARIES	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

Other: WILLIAMSBURG COUNTY DISABILITIES & SPECIAL NEEDS BOARD	true

Other: KERSHAW HEALTH DBA THE HEALTH SERVICES DISTRICT OF KERSHAW CO	true

Other: TOWN OF AWENDAW SOUTH CAROLINA	true

Other: SANTEE-LYNCHES REGIONAL COUNCIL OF GOVERNMENTS	true

Other: SANTEE COOPER AUTHORITY	true

Other: SPARTANBURG COUNTY SC TREASURER	true

Other: FLORENCE COUNTY SC TREASURER	true

Other: CITY OF NORTH CHARLESTON SC	true

Other: SYNOVUS TRUST COMPANY, N.A.	true

Other: CITY OF LORIS SC	true

Other: GEORGIA HAP ADMINISTRATION AKA NATIONAL HSG COMPLIANCE	true

Other: HOUSING AUTHORITY OF BREMEN GA	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: STATE TREASURER OF ALABAMA	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: SYNOVUS TRUST COMPANY, N.A.	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: PNB TRUST DEPT	true

null	false

Other: REPURCHASE AGREEMENT	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: STATE TREASURER OF FLORIDA	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: NORTH CAROLINA STATE TREASURER	true

Other: REPURCHASE AGREEMENT	true

Other: STATE TREASURER OF FLORIDA	true

null	false

null	false

Other: RICHLAND COUNTY SC TREASURER	true

Other: SYNOVUS TRUST COMPANY, N.A.	true

Other: SOUTH CAROLINA STATE TREASURER - DEDICATED	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: SANTEE COOPER AUTHORITY	true

Other: CITY OF NORTH MYRTLE BEACH SC	true

Other: RICHLAND COUNTY SC TREASURER	true

Other: SPARTANBURG COUNTY SC PUBLIC LIBRARIES	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

null	false

Other: VA TREASURY BOARD	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: METROPOLITAN WASHINGTON AIRPORTS AUTHORITY	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

null	false

null	false

null	false

Other: HOUSING AUTHORITY OF CITY OF FORT VALLEY	true

null	false

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

Other: REPURCHASE AGREEMENT	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: STATE TREASURER OF FLORIDA	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: COLLIER'S INTERNATIONAL FBO CITY OF COLLEGE PARK SC	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

Other: HOUSING AUTHORITY FORT WALTON BEACH FL	true

Other: HOUSING AUTHORITY WALKER COUNTY AL	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: REPURCHASE AGREEMENT	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

Other: HOUSING AUTHORITY OF CITY OF FORT VALLEY	true

Other: RICHLAND COUNTY SC TREASURER	true

null	false

Other: CHARLESTON COUNTY SCHOOL DISTRICT	true

null	false

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: CITY OF MYRTLE BEACH	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

Other: SOUTH CAROLINA STATE TREASURER - DEDICATED	true

Other: FLORENCE COUNTY SC TREASURER	true

Other: NORTHWEST GA HOUSING AUTHORITY	true

null	false

null	false

Other: THE HOUSING COMM OF ANNE ARUNDEL COUNT	true

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF ALABAMA	true

Other: SANTEE COOPER AUTHORITY	true

Other: CHARLESTON CTY SC HUMAN SVS DBA PALMETO COMM ACTION	true

Other: SYNOVUS TRUST COMPANY, N.A.	true

Other: RICHLAND COUNTY SC TREASURER	true

Other: SPARTANBURG COUNTY SC PUBLIC LIBRARIES	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF FLORIDA	true

null	false

To:	true

To:	true

Other: STATE TREASURER OF FLORIDA	true

Other: STATE OF TENNESSEE	true

Other: METTER HOUSING AUTHORITY	true

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: STATE TREASURER OF ALABAMA	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: STATE TREASURER OF FLORIDA	true

null	false

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: STATE TREASURER OF FLORIDA	true

Other: ASHEBORO HOUSING AUTHORITY	true

Other: CITY OF ALBEMARLE	true

null	false

null	false

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

Other: REPURCHASE AGREEMENT	true

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: THE COTTON BOARD	true

Other: THE COTTON BOARD	true

Other: OPIOIDS SETTLEMENT FUND QSF DISTRIBUTION	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: SANTEE COOPER AUTHORITY	true

null	false

Other: HOUSING AUTHORITY FORT WALTON BEACH FL	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

null	false

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

Other: CHARLESTON COUNTY SCHOOL DISTRICT	true

Other: ASHEBORO HOUSING AUTHORITY	true

null	false

null	false

Other: PNB TRUST DEPT	true

Other: STATE TREASURER OF FLORIDA	true

Other: CITY OF NORTH MYRTLE BEACH SC	true

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: REPURCHASE AGREEMENT	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: STATE TREASURER OF FLORIDA	true

Other: PNB TRUST DEPT	true

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

null	false

Other: REIDVILLE AREA FIRE DISTRICT	true

Other: SANTEE COOPER AUTHORITY	true

Other: CHARLESTON CTY SC HUMAN SVS DBA PALMETO COMM ACTION	true

Other: HOUSING AUTHORITY OF CITY OF FORT VALLEY	true

null	false

Other: SYNOVUS TRUST COMPANY, N.A.	true

Other: SANTEE COOPER AUTHORITY	true

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: CITY OF MYRTLE BEACH	true

Other: PNB TRUST DEPT	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: CITY OF MYRTLE BEACH	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: STATE TREASURER OF ALABAMA	true

Other: PNB TRUST DEPT	true

Other: CITY OF MYRTLE BEACH	true

Other: RICHLAND COUNTY SC TREASURER	true

null	false

Other: STATE TREASURER OF ALABAMA	true

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: REPURCHASE AGREEMENT	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: FLORENCE COUNTY SC TREASURER	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

Other: STATE TREASURER OF ALABAMA	true

Other: SYNOVUS TRUST COMPANY, N.A.	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: UNIVERSITY OF KENTUCKY	true

Other: THOMASVILLE HOUSING AUTHORITY	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

Other: WELLS FARGO CME - MARGIN	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: METROPOLITAN WASHINGTON AIRPORTS AUTHORITY	true

null	false

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: REPURCHASE AGREEMENT	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

null	false

Other: REPURCHASE AGREEMENT	true

Other: CITY OF MYRTLE BEACH	true

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF FLORIDA	true

null	false

null	false

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

null	false

Other: THOMASVILLE HOUSING AUTHORITY	true

null	false

Other: REPURCHASE AGREEMENT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: VA TREASURY BOARD	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: SANTEE COOPER AUTHORITY	true

Other: HOUSING AUTHORITY FORT WALTON BEACH FL	true

Other: SYNOVUS TRUST COMPANY, N.A.	true

Other: HOUSING AUTHORITY ALBANY GA	true

Other: THOMASVILLE HOUSING AUTHORITY	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

null	false

Other: HOUSING AUTHORITY OF BOWDON GEORGIA	true

Other: COLLIER'S INTERNATIONAL FBO CITY OF COLLEGE PARK SC	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: SOUTH CAROLINA STATE TREASURER - DEDICATED	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

Other: KERSHAW HEALTH DBA THE HEALTH SERVICES DISTRICT OF KERSHAW CO	true

Other: HOUSING AUTHORITY CRESTVIEW FL	true

Other: SANTEE-LYNCHES REGIONAL COUNCIL OF GOVERNMENTS	true

Other: LEE COUNTY SC DSN BOARD	true

Other: CITY OF LORIS SC	true

Other: FLORENCE COUNTY SC TREASURER	true

Other: SANTEE COOPER AUTHORITY	true

Other: STATE TREASURER OF ALABAMA	true

null	false

Other: HOUSING AUTHORITY ALBANY GA	true

Other: HOUSING AUTHORITY FORT WALTON BEACH FL	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: SOLID WASTE  DISPOSAL AUTHORITY - HUNTSVILLE AL	true

Other: NORTHWEST GA HOUSING AUTHORITY	true

Other: STATE TREASURER OF FLORIDA	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: NORTH CAROLINA STATE TREASURER	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

null	false

null	false

null	false

null	false

Other: CITY OF MYRTLE BEACH	true

null	false

null	false

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: RICHLAND COUNTY SC TREASURER	true

Other: CITY OF NORTH MYRTLE BEACH SC	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

null	false

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: CITY OF MYRTLE BEACH	true

Other: CITY OF MYRTLE BEACH	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

Other: SYNOVUS TRUST COMPANY, N.A.	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: CITY OF MYRTLE BEACH	true

Other: STATE TREASURER OF ALABAMA	true

Other: CITY OF MYRTLE BEACH	true

Other: THOMASVILLE HOUSING AUTHORITY	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: CHARLESTON CTY SC HUMAN SVS DBA PALMETO COMM ACTION	true

Other: FLORENCE COUNTY SC TREASURER	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: COLLIER'S INTERNATIONAL FBO CITY OF COLLEGE PARK SC	true

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF SOUTH CAROLINA - POOL	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF ALABAMA	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: UNIVERSITY OF KENTUCKY	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

null	false

Other: THOMASVILLE HOUSING AUTHORITY	true

Other: REIDVILLE AREA FIRE DISTRICT	true

Other: TOWN OF MCCLELLANVILLE	true

Other: THE HOUSING COMM OF ANNE ARUNDEL COUNT	true

Other: ASHEBORO HOUSING AUTHORITY	true

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: KERSHAW HEALTH DBA THE HEALTH SERVICES DISTRICT OF KERSHAW CO	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: CITY OF NORTH MYRTLE BEACH SC	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

Other: HOUSING AUTHORITY OF CITY OF FORT VALLEY	true

Other: CITY OF NORTH CHARLESTON SC	true

Other: SANTEE COOPER AUTHORITY	true

Other: SPARTANBURG COUNTY SC PUBLIC LIBRARIES	true

null	false

Other: REPURCHASE AGREEMENT	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: PNB TRUST DEPT	true

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: STATE TREASURER OF FLORIDA	true

Other: STATE TREASURER OF FLORIDA	true

Other: PNB TRUST DEPT	true

null	false

Other: NORTH CAROLINA STATE TREASURER	true

null	false

Other: NORTH CAROLINA STATE TREASURER	true

Other: NORTH CAROLINA STATE TREASURER	true

null	false

Other: PNB TRUST DEPT	true

Other: HOUSING AUTHORITY OF COLUMBUS GA	true

Other: NORTHWEST GA HOUSING AUTHORITY	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

null	false

null	false

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: COLLIER'S INTERNATIONAL FBO CITY OF COLLEGE PARK SC	true

Other: STATE TREASURER OF ALABAMA	true

Other: PNB TRUST DEPT	true

Other: REPURCHASE AGREEMENT	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

Other: CITY OF MYRTLE BEACH	true

Other: STATE TREASURER OF FLORIDA	true

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: STATE TREASURER OF FLORIDA	true

Other: KERSHAW HEALTH DBA THE HEALTH SERVICES DISTRICT OF KERSHAW CO	true

Other: HORRY COUNTY SC TREASURER	true

Other: COLLIER'S INTERNATIONAL FBO CITY OF COLLEGE PARK SC	true

null	false

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: GEORGIA SECURED DEPOSIT PROGRAM	true

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: US TRUSTEE R-6 DALLAS	true

Other: US TRUSTEE CHAPTER 13 BANKRUPTCY ADMINISTRATOR TUSC, AL	true

Other: US TRUSTEE R-21 MIAMI	true

Other: US TRUSTEE R-21 ATLANTA	true

Other: US TRUSTEE R-21 TAMPA	true

Other: US TRUSTEE EASTERN DISTRICT NY	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: BURLINGTON HOUSING AUTHORITY	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: WILLIAMSBURG COUNTY DISABILITIES & SPECIAL NEEDS BOARD	true

Other: SYNOVUS TRUST COMPANY, N.A.	true

Other: TOWN OF AWENDAW SOUTH CAROLINA	true

Other: HOUSING AUTHORITY WALKER COUNTY AL	true

Other: NORTHWEST GA HOUSING AUTHORITY	true

Other: SOUTH CAROLINA STATE TREASURER - DEDICATED	true

Other: HOUSING AUTHORITY OF MACON-BIBB COUNTY	true

null	false

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

Other: GEORGIA HAP ADMINISTRATION AKA NATIONAL HSG COMPLIANCE	true

Other: LEE COUNTY SC DSN BOARD	true

Other: SANTEE-LYNCHES REGIONAL COUNCIL OF GOVERNMENTS	true

Other: RICHLAND COUNTY SC CLERK OF COURT	true

Other: METTER HOUSING AUTHORITY	true

Other: CITY OF NORTH MYRTLE BEACH SC	true

Other: KERSHAW HEALTH DBA THE HEALTH SERVICES DISTRICT OF KERSHAW CO	true

Other: SANTEE COOPER AUTHORITY	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: STATE TREASURER OF FLORIDA	true

Other: FLORENCE COUNTY SC TREASURER	true

Other: GEORGIA HAP ADMINISTRATION AKA NATIONAL HSG COMPLIANCE	true

Other: MIDLANDS TECHNICAL COLLEGE	true

Other: LEE COUNTY SC DSN BOARD	true

Other: NORTHWEST GA HOUSING AUTHORITY	true

Other: TOWN OF AWENDAW SOUTH CAROLINA	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

Other: STATE TREASURER OF FLORIDA	true

Other: CITY OF LORIS SC	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

Other: REPURCHASE AGREEMENT	true

null	false

Other: COLLIER'S INTERNATIONAL FBO CITY OF COLLEGE PARK SC	true

Other: KERSHAW HEALTH DBA THE HEALTH SERVICES DISTRICT OF KERSHAW CO	true

Other: SPARTANBURG COUNTY SC PUBLIC LIBRARIES	true

Other: RICHLAND COUNTY SC TREASURER	true

Other: STATE TREASURER OF FLORIDA	true

Other: SANTEE COOPER AUTHORITY	true

Other: HORRY COUNTY SC TREASURER	true

Other: FIRST COMMERCIAL BANK BIRMINGHAM CORP TRUST	true

Other: STATE TREASURER OF FLORIDA	true

Other: STATE TREASURER OF FLORIDA	true

null	false

Other: STATE TREASURER OF FLORIDA	true

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

null	false

null	false

Other: CITY OF LORIS SC	true

Other: GEORGIA HAP ADMINISTRATION AKA NATIONAL HSG COMPLIANCE	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: CITY OF MYRTLE BEACH	true

null	false

null	false

Other: ADVANCED TECHNOLOGY INTERNATIONAL	true

Other: STATE TREASURER OF FLORIDA	true

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

Other: STATE TREASURER OF FLORIDA	true

Other: STATE TREASURER OF ALABAMA	true

Other: REPURCHASE AGREEMENT	true

null	false

null	false

null	false

Other: STATE TREASURER OF ALABAMA	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: US TRUSTEE EASTERN DISTRICT NY	true

Other: US TRUSTEE R-6 DALLAS	true

Other: US TRUSTEE R-21 ATLANTA	true

Other: US TRUSTEE R-9 DETROIT	true

Other: US TRUSTEE R-21 TALLAHASSEE	true

Other: US TRUSTEE R-21 TAMPA	true

Other: US TRUSTEE CHAPTER 13 BANKRUPTCY ADMINISTRATOR TUSC, AL	true

Other: US TRUSTEE R-21 MIAMI	true

null	false

null	false

null	false

Other: REPURCHASE AGREEMENT	true

null	false

Other: REPURCHASE AGREEMENT	true

Other: US TRUSTEE R-10 INDIANAPOLIS	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

Other: PNB TRUST DEPT	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

Other: PNB TRUST DEPT	true

Other: CITY OF MYRTLE BEACH	true

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false

null	false
 

In [0]:
%pip install openpyxl 
import os
import pandas as pd
import openpyxl


src_dir = "/Volumes/finance/staging/imports/legacy_pnfp/other"
pattern = "Security Data File"
target_cols_upper = ["CURRENT PERIOD RUN DATE", "MARKET VALUE", "PERCENT PLEDGED"]

files = sorted([
    f for f in os.listdir(src_dir)
    if f.endswith(".xlsx") and pattern in f
    and len(f.split()) >= 5 and f.split()[1].isdigit() and f.split()[0].isalpha()
])

dfs = []
for file in files:
    file_path = os.path.join(src_dir, file)
    try:
        # Detect header row & sheet: try default sheet and 'Data' sheet, header=9 and header=0
        header_row = None
        sheet_name = None
        for sn in [0, 'Data']:  # 0 = default first sheet
            for h in [9, 0]:
                try:
                    df_peek = pd.read_excel(file_path, sheet_name=sn, header=h, dtype=str, nrows=0)
                    col_map = {str(c).upper().strip(): c for c in df_peek.columns}
                    if all(t in col_map for t in target_cols_upper):
                        header_row = h
                        sheet_name = sn
                        break
                except Exception:
                    continue
            if header_row is not None:
                break

        if header_row is None:
            print(f"⚠ Skipping {file}: target columns not found at header=9 or header=0 on any sheet")
            continue

        # Read ALL columns from the detected sheet
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row, dtype=str)

        # Standardize ALL column names to uppercase with underscores
        df.columns = [str(c).upper().strip().replace(' ', '_') for c in df.columns]

        # Type-cast specific columns; leave all others as string
        df['CURRENT_PERIOD_RUN_DATE'] = pd.to_datetime(df['CURRENT_PERIOD_RUN_DATE'], errors='coerce')
        df['MARKET_VALUE'] = pd.to_numeric(df['MARKET_VALUE'], errors='coerce').astype('float64')
        df['PERCENT_PLEDGED'] = pd.to_numeric(df['PERCENT_PLEDGED'], errors='coerce').astype('float64')

        df['source_file'] = file
        dfs.append(df)
        print(f"✓ Loaded {file}: {len(df):,} rows  (header={header_row})")
    except Exception as e:
        print(f"✗ Error reading {file}: {e}")

if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    spark_df = spark.createDataFrame(combined_df)
    spark_df.createOrReplaceTempView("pnfp_security_data_file")



    print(f"\nTotal rows in temp view: {spark_df.count():,}")
    display(spark_df)
else:
    print("No files matched or could be read.")


excel_path = "/Volumes/finance/staging/imports/legacy_pnfp/other/pnfp_sec_level_ref.xlsx"
df = pd.read_excel(excel_path)
spark_df = spark.createDataFrame(df)
spark_df.createOrReplaceTempView("pnfp_sec_level_ref")
display(spark_df)



In [0]:
# Regex replace () with _ in column names
import re
combined_df.columns = [re.sub(r'[()]', '_', col) for col in combined_df.columns]

spark_df = spark.createDataFrame(combined_df)
spark_df.write.mode("append").saveAsTable("finance.staging.lpnfp_isp")

In [0]:
%sql
select distinct CURRENT_PERIOD_RUN_DATE from finance.staging.lpnfp_isp

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW pnfp_pledge_code_map AS
    SELECT
      'Bankruptcy' AS pledge_code,
      'BANKRUPTCY TRUSTEE' AS pledge_desc
    UNION ALL
    SELECT
      'ATI',
      'ADVANCED TECHNOLOGY INTERNATIONAL'
    UNION ALL
    SELECT
      'PSE',
      'TENNESSEE STATE COLLATERAL POOL'
    UNION ALL
    SELECT
      'PSE',
      'METROPOLITAN WASHINGTON AIRPORTS AUTHORITY'
    UNION ALL
    SELECT
      'PSE',
      'STATE OF GEORGIA'
    UNION ALL
    SELECT
      'PSE',
      'UNIVERSITY OF KENTUCKY'
    UNION ALL
    SELECT
      'PSE',
      'OPIOIDS SETTLEMENT FUND QSF DISTRIBUTION'
    UNION ALL
    SELECT
      'PSE',
      'FLORIDA DEPARTMENT OF FINANCIAL SERVICES'
    UNION ALL
    SELECT
      'PSE',
      'NORTH CAROLINA STATE TREASURER'
    UNION ALL
    SELECT
      'CollateralSwap',
      'GOLDMAN SACHS'
    UNION ALL
    SELECT
      'CollateralSwap',
      'WELLS FARGO'
    UNION ALL
    SELECT
      'PNB Trust',
      'PNB TRUST DEPT'
    UNION ALL
    SELECT
      'PSE',
      'CITY OF MYRTLE BEACH'
    UNION ALL
    SELECT
      'PSE',
      'CITY OF ALBEMARLE'
    UNION ALL
    SELECT
      'PSE',
      'THOMASVILLE HOUSING AUTHORITY'
    UNION ALL
    SELECT
      'PSE',
      'THE HOUSING COMM OF ANNE ARUNDEL COUNT'
    UNION ALL
    SELECT
      'PSE',
      'ASHEBORO HOUSING AUTHORITY'
    UNION ALL
    SELECT
      'PSE',
      'REIDVILLE AREA FIRE DISTRICT'
    UNION ALL
    SELECT
      'PSE',
      'Town of McClellanville'
    UNION ALL
    SELECT
      'PSE',
      'CHARLESTON COUNTY SCHOOL DISTRICT'
    UNION ALL
    SELECT
      'PSE',
      'BURLINGTON HOUSING AUTHORITY'
    UNION ALL
    SELECT
      'Repo',
      'REPURCHASE AGREEMENT'
    UNION ALL
    SELECT
      'Opioid',
      'THE COTTON BOARD'
    UNION ALL
    SELECT
      'PSE',
      'VA TREASURY BOARD'
    UNION ALL
    SELECT
      'FHLB',
      'FEDERAL HOME LOAN BANK'
      
    UNION ALL
    SELECT
    'unmapped',
    'unmapped'
    UNION ALL
    SELECT
      'PSE',
      'HOUSING AUTHORITY CITY OF HIGH POINT NC'
      UNION ALL
    SELECT
      'PSE',
      'TOWN OF MCCLELLANVILLE'


;



CREATE OR REPLACE TEMP VIEW PLEDGEE_LOOKUP AS
select distinct
  ps.DESCRIPTION,
  ps.cusip,
  ps.trans_no,
  ps.percent_pledged,
  ps.PLEDGED_ENTITY,
  ps.PLEDGED_GROUP,
  ps.lot,
  ps.market_value

FROM
  finance.staging.lpnfp_isp ps
   WHERE
  ps.MARKET_VALUE is NOT NULL
  AND ps.PLEDGED_ENTITY IS NOT NULL



UNION ALL
--CREATE OR REPLACE TEMP VIEW PLEDGEE_LOOKUP_UNMAPPED AS
select
distinct
  ps.DESCRIPTION,
  ps.cusip,
  ps.trans_no,
  ps.percent_pledged,
  'unmapped' as PLEDGED_ENTITY,
  'unmapped' as PLEDGED_GROUP,
  ps.lot,
  ps.market_value

FROM
  finance.staging.lpnfp_isp ps

WHERE
  ps.MARKET_VALUE is NOT NULL
  AND ps.PLEDGED_ENTITY IS NULL
  
  
  ;


CREATE OR REPLACE TEMP VIEW pnfp_mv AS 


SELECT 
    ps.CURRENT_PERIOD_RUN_DATE,
    ps.DESCRIPTION,
    ps.cusip,
    ps.trans_no,
    ps.percent_pledged,
    ps.lot,

    CASE 
    WHEN 
    pl.pledged_group= 'unmapped' 
    THEN 'unmapped'
    WHEN 
    ps.PLEDGED_ENTITY IS NULL 
    THEN PL.PLEDGED_ENTITY ELSE ps.PLEDGED_ENTITY END AS PLEDGED_ENTITY,


    CASE   
    WHEN pl.pledged_entity = 'unmapped' THEN 'unmapped'
    WHEN ps.PLEDGED_GROUP IS NULL THEN PL.PLEDGED_GROUP
    ELSE ps.PLEDGED_GROUP END AS PLEDGED_GROUP,
PS.MARKET_VALUE,

    CASE WHEN 
    pl.pledged_entity = 'unmapped' THEN TRY_CAST(ps.market_value_pledged AS DOUBLE)
    WHEN TRY_CAST(ps.MARKET_VALUE_PLEDGED AS DOUBLE) IS NULL THEN (ps.percent_pledged/100) * ps.MARKET_VALUE
    ELSE TRY_CAST(ps.MARKET_VALUE_PLEDGED AS DOUBLE) END AS MARKET_VALUE_PLEDGED,
    map.level

FROM finance.staging.lpnfp_isp ps
    JOIN finance.default.dim_dates d
        ON d.date = cast(ps.current_period_run_date as date)
    LEFT JOIN pledgee_lookup pl
        ON ps.cusip = pl.cusip
        AND ps.DESCRIPTION = pl.DESCRIPTION 
        AND ps.trans_no = pL.trans_no
        AND ps.percent_pledged = pL.percent_pledged
        AND ps.lot = pl.lot
        AND pl.market_value = ps.market_value
    LEFT JOIN pnfp_sec_level_ref map
        ON ps.DESCRIPTION = map.description

WHERE pl.DESCRIPTION IS NOT null
  AND d.date = d.End_of_Month;


CREATE OR REPLACE TEMP VIEW pnfp_mv_by_level AS
SELECT
  mv.*,
  pm.pledge_code

FROM
  pnfp_mv mv
   -- LEFT 
    JOIN pnfp_pledge_code_map pm
      ON mv.PLEDGED_ENTITY = pm.PLEDGE_desc;
SELECT * FROM finance.staging.lpnfp_isp;


CREATE OR REPLACE TEMP VIEW unmapped AS
SELECT
  *

FROM
  pnfp_security_data_file

WHERE
  trans_no IN (
    SELECT DISTINCT
      trans_no

    FROM
      pnfp_mv

    WHERE
      pledged_entity = 'unmapped'
  );


CREATE OR REPLACE TEMP VIEW CONSTANTS AS
SELECT DISTINCT
  trans_no,


CASE
    WHEN pledged_entity = 'PNB TRUST DEPT' THEN 'PNB Trust'
    WHEN pledged_entity = 'TENNESSEE STATE COLLATERAL POOL' THEN 'Public Fund'
    WHEN pledged_entity = 'ADVANCED TECHNOLOGY INTERNATIONAL' THEN 'Quasi Governmental-Non public fund'
    WHEN pledged_entity = 'THOMASVILLE HOUSING AUTHORITY' THEN 'Public Fund'
        ELSE pledged_group
            end as  pledged_group,

  pledged_entity

FROM
  unmapped

WHERE
  pledged_entity IS NOT NULL
  AND (pledged_entity <> '0'
  and pledged_group <> '0');


SELECT
  *

FROM
  CONSTANTS;

  
CREATE OR REPLACE TEMP VIEW mapped_unmapped AS 
SELECT 
    pm.CURRENT_PERIOD_RUN_DATE,
    pm.DESCRIPTION,
    pm.cusip,
    pm.trans_no,
    pm.percent_pledged,
    pm.lot,
    c.PLEDGED_ENTITY,
    c.PLEDGED_GROUP,
    pm.MARKET_VALUE,

    CASE WHEN
        PM.MARKET_VALUE_PLEDGED IS NULL THEN (pm.percent_pledged/100) * pm.market_value
            ELSE pm.MARKET_VALUE_PLEDGED
                END AS MARKET_VALUE_PLEDGED,
    pm.level
FROM 
    pnfp_mv pm 
JOIN constants c
    ON c.trans_no = pm.trans_no 
WHERE 
    pm.pledged_entity = 'unmapped' ;

CREATE OR REPLACE TEMP VIEW pnfp_mv_by_level_2 AS
SELECT
  mv.*,
  pm.pledge_code

FROM
  mapped_unmapped mv
   -- LEFT 
    JOIN pnfp_pledge_code_map pm
      ON mv.PLEDGED_ENTITY = pm.PLEDGE_desc;




In [0]:
%sql
CREATE OR REPLACE TEMP VIEW mapped_unmapped AS 
SELECT 
    pm.CURRENT_PERIOD_RUN_DATE,
    pm.DESCRIPTION,
    pm.cusip,
    pm.trans_no,
    pm.percent_pledged,
    pm.lot,
    c.PLEDGED_ENTITY,
    c.PLEDGED_GROUP,
    pm.MARKET_VALUE,

    CASE WHEN
        PM.MARKET_VALUE_PLEDGED IS NULL THEN (pm.percent_pledged/100) * pm.market_value
            ELSE pm.MARKET_VALUE_PLEDGED
                END AS MARKET_VALUE_PLEDGED,
    pm.level
FROM 
    pnfp_mv pm 
JOIN constants c
    ON c.trans_no = pm.trans_no 
WHERE 
    pm.pledged_entity = 'unmapped' ;

CREATE OR REPLACE TEMP VIEW pnfp_mv_by_level_2 AS
SELECT
  mv.*,
  pm.pledge_code

FROM
  mapped_unmapped mv
   -- LEFT 
    JOIN pnfp_pledge_code_map pm
      ON mv.PLEDGED_ENTITY = pm.PLEDGE_desc;

In [0]:
%sql
      
CREATE OR REPLACE TEMP VIEW pnfp_mv_by_level_f AS 
SELECT current_period_run_date, pledge_code, level, TRY_CAST(market_value_pledged AS DOUBLE) AS MV FROM pnfp_mv_by_level WHERE pledge_code <> 'unmapped' 
UNION ALL
SELECT current_period_run_date, pledge_code, level, TRY_CAST(market_value_pledged AS DOUBLE) AS MV FROM pnfp_mv_by_level_2
;

select 

CAST(current_period_run_date as DATE) eomdate,
'PNFP' as legacy_bank,
CASE WHEN pledge_code IS NOT NULL THEN pledge_code ELSE NULL END as pledgee_type,
level as hqla_level,
SUM(TRY_CAST(mv AS DOUBLE)) as mv

from pnfp_mv_by_level_f
    GROUP BY 1,2,3,4

Databricks visualization. Run in Databricks to view.